# 11 — K Selection for Unlabeled Datasets

Selects the best number of archetypes (K) from a PAE sweep without requiring cell type labels.

**Configured for:** breast CyTOF dataset (`normalized_not_scaled`, 15 samples, ~1.1M cells)  
**Config:** `configs/breast_pae_k_sweep.yaml` — sweeps K ∈ {6, 8, 10, 12, 14, 16}, `downsample_factor=10` (~113k cells)

**Workflow:**
1. **Data prep** (cell below) — merge parquet files → `data/breast_cytof_processed.h5ad`
2. **Run sweep** — `python scripts/run_experiment_suite.py --config configs/breast_pae_k_sweep.yaml`
3. **Analysis** — run remaining cells to get K-selection diagnostics and recommendation

In [22]:
from pathlib import Path
import sys

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root')

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Repo root:', REPO_ROOT)

Repo root: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv


## Step 1 — Prepare Breast Cancer Data

Merges all `normalized_not_scaled_*.parquet` files (excluding `.2` replicate samples) into
`data/breast_cytof_processed.h5ad`. Skips if the file already exists.

- Excludes backbone markers: `H3`, `H4`, `H3.3`
- Clips each marker to `[0, 99.9th percentile]`
- Stores `sample_id` in `obs` (no cell type labels — unlabeled dataset)

In [23]:
import re
import numpy as np
import pandas as pd

# ── Data prep config ──────────────────────────────────────────────────────────
BREAST_DATA_DIR  = Path('/Users/ronguy/Dropbox/CyTOF_Breast/for_guy')
FILE_VARIANT     = 'normalized_not_scaled'
EXCLUDE_DOT2     = True                   # skip .2 samples (replicates)
EXCLUDED_MARKERS = {'H3', 'H4', 'H3.3'}  # backbone markers
CLIP_UPPER_PCT   = 99.9
SAMPLE_ID_REGEX  = r'_([0-9]+(?:\.[0-9]+)?)$'
OUTPUT_H5AD      = REPO_ROOT / 'data' / 'breast_cytof_processed.h5ad'
# ─────────────────────────────────────────────────────────────────────────────

if OUTPUT_H5AD.exists():
    print(f'h5ad already exists: {OUTPUT_H5AD}')
    print('Delete it and re-run this cell to regenerate.')
else:
    try:
        import anndata as ad
    except ImportError:
        raise ImportError('anndata required: pip install anndata')

    files = sorted(BREAST_DATA_DIR.glob(f'{FILE_VARIANT}_*.parquet'))
    assert files, f'No parquet files found in {BREAST_DATA_DIR}'
    print(f'Found {len(files)} parquet files.')

    # First pass: find markers common to ALL included samples
    # (avoids NaN columns from samples that lack a marker)
    common_markers = None
    for fp in files:
        m = re.search(SAMPLE_ID_REGEX, fp.stem)
        sid = m.group(1) if m else fp.stem
        if EXCLUDE_DOT2 and sid.endswith('.2'):
            continue
        df = pd.read_parquet(fp)
        cols = set(c for c in df.columns if c not in EXCLUDED_MARKERS
                   and pd.api.types.is_numeric_dtype(df[c]))
        common_markers = cols if common_markers is None else common_markers & cols
    marker_cols = sorted(common_markers)
    print(f'Common markers ({len(marker_cols)}): {marker_cols}')

    # Second pass: load with common markers only
    dfs = []
    for fp in files:
        m = re.search(SAMPLE_ID_REGEX, fp.stem)
        sid = m.group(1) if m else fp.stem
        if EXCLUDE_DOT2 and sid.endswith('.2'):
            print(f'  Skipping {fp.name}  (replicate)')
            continue
        df = pd.read_parquet(fp)
        df['sample_id'] = sid
        dfs.append(df[marker_cols + ['sample_id']])
        print(f'  Loaded  {fp.name}  ({len(df):,} cells)')

    combined = pd.concat(dfs, ignore_index=True)
    print(f'\nTotal cells: {len(combined):,}  |  NaN in markers: {combined[marker_cols].isna().sum().sum()}')

    X = combined[marker_cols].values.astype('float32')
    upper = np.percentile(X, CLIP_UPPER_PCT, axis=0)
    X = np.clip(X, 0, upper)

    obs = pd.DataFrame({'cell_id': [f'cell_{i}' for i in range(len(combined))],
                        'sample_id': combined['sample_id'].values})
    obs.index = obs['cell_id'].values

    adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=marker_cols))
    OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(OUTPUT_H5AD)
    print(f'\nSaved: {OUTPUT_H5AD}  ({adata.shape[0]:,} cells x {adata.shape[1]} markers)')

h5ad already exists: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/data/breast_cytof_processed.h5ad
Delete it and re-run this cell to regenerate.


## Step 2 — Run the K Sweep

The cell below runs the sweep directly. Set `RUN_SWEEP = False` to skip if you've already run it.

Alternatively, run from terminal:
```bash
python scripts/run_experiment_suite.py --config configs/breast_pae_k_sweep.yaml
```

In [ ]:
import os
import time
from cytof_archetypes.experiments import load_suite_config, run_experiment_suite

CONFIG_PATH = REPO_ROOT / 'configs' / 'breast_pae_k_sweep.yaml'
assert CONFIG_PATH.exists(), f'Missing config: {CONFIG_PATH}'

RUN_SWEEP = True   # set to False to skip and go straight to analysis

cfg = load_suite_config(CONFIG_PATH)
_env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
if _env:
    cfg['output_dir'] = _env

# Enable live loss plot
cfg['show_progress'] = True
cfg['show_run_logs'] = True
cfg['show_training_progress'] = True
pae_cfg = cfg.setdefault('methods', {}).setdefault('probabilistic_archetypal_ae', {})
pae_cfg['live_loss_plot'] = False  # tqdm status bar shows loss instead
pae_cfg['show_training_progress'] = True

print('Config:', CONFIG_PATH)
print('Output dir:', cfg['output_dir'])
print('K sweep:', cfg.get('sweeps', {}).get('k_values', []))
print('Seeds:', cfg.get('seeds', []))
print('Downsample factor:', cfg.get('dataset', {}).get('downsample_factor', 'none'))

if RUN_SWEEP:
    t0 = time.time()
    out_dir = run_experiment_suite(cfg)
    print(f'Sweep completed in {(time.time()-t0)/60:.1f} min  |  output: {out_dir}')
else:
    print('Sweep skipped (RUN_SWEEP=False). Using existing outputs.')

Config: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/configs/breast_pae_k_sweep.yaml
Output dir: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/outputs/breast_pae_k_sweep
K sweep: [8]
Seeds: [42]
Downsample factor: none


Suite phases:   0%|          | 0/12 [00:00<?, ?phase/s]

[suite-phase] 1/12: prepare_data
[suite-phase] 2/12: run_core_benchmark (all methods × dims × seeds)


MPS/GPU benchmark runs:   0%|          | 0/1 [00:00<?, ?run/s]

[core-stage] running 1 MPS/GPU+serial jobs
[core-run 1/1] START method=probabilistic_archetypal_ae dim=8 seed=42 device=mps


probabilistic_archetypal_ae dim=8 seed=42 [epoch]:   0%|          | 0/1000 [00:00<?, ?epoch/s]

In [ ]:
import os

# ── USER CONFIG ───────────────────────────────────────────────────────────────
OUTPUT_DIR    = REPO_ROOT / 'outputs' / 'breast_pae_k_sweep'
METHOD        = 'probabilistic_archetypal_ae'
K_VALUES      = None    # None = auto-discover from dim_* subdirectories
VAR_THRESH    = 1e5     # mean archetype variance above this = dead
ASSIGN_THRESH = 0.25    # max_weight threshold for 'assigned' cell
# ─────────────────────────────────────────────────────────────────────────────

_env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
if _env:
    OUTPUT_DIR = Path(_env)

RUNS_DIR = OUTPUT_DIR / 'runs' / METHOD
if not RUNS_DIR.exists():
    print(f'Runs directory not found: {RUNS_DIR}')
    print('Run the sweep first (Step 2 above).')
else:
    if K_VALUES is None:
        K_VALUES = sorted([
            int(p.name.split('_')[1])
            for p in RUNS_DIR.iterdir()
            if p.is_dir() and p.name.startswith('dim_') and p.name.split('_')[1].isdigit()
        ])
    print(f'Output dir:  {OUTPUT_DIR}')
    print(f'K values:    {K_VALUES}')

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
from itertools import combinations


def _load_run(k, seed):
    # find the actual directory regardless of zero-padding (dim_4 vs dim_04)
    dim_matches = [p for p in RUNS_DIR.iterdir()
                   if p.is_dir() and p.name.startswith('dim_')
                   and p.name.split('_')[-1].isdigit()
                   and int(p.name.split('_')[-1]) == k]
    if not dim_matches: return None
    run_dir = dim_matches[0] / f'seed_{seed}'
    required = [run_dir/'component_means.csv', run_dir/'component_vars.csv', run_dir/'training_history.csv']
    weight_paths = [run_dir/'val'/'weights.csv', run_dir/'test'/'weights.csv', run_dir/'train'/'weights.csv']
    if not all(p.exists() for p in required): return None
    if not any(p.exists() for p in weight_paths): return None
    means = pd.read_csv(run_dir/'component_means.csv', index_col=0)
    cvars = pd.read_csv(run_dir/'component_vars.csv', index_col=0)
    hist  = pd.read_csv(run_dir/'training_history.csv')
    wframes = []
    for p in weight_paths:
        if p.exists():
            df = pd.read_csv(p, index_col=0)
            wframes.append(df[[c for c in df.columns if c.startswith('w_')]])
    W = pd.concat(wframes).values.astype('float32')
    return dict(k=k, seed=seed, means=means, cvars=cvars, hist=hist, W=W)


def _compute_metrics(run):
    k, means, cvars, hist, W = run['k'], run['means'], run['cvars'], run['hist'], run['W']
    best_val_loss = hist['val_loss'].min()
    n_epochs      = len(hist)
    mean_var      = cvars.mean(axis=1).values
    n_active      = int((mean_var < VAR_THRESH).sum())
    W_safe        = np.clip(W, 1e-9, 1.0)
    entropy       = -(W_safe * np.log(W_safe)).sum(axis=1)
    effective_k   = float(np.exp(entropy.mean()))
    M             = means.values
    diffs         = [np.linalg.norm(M[i]-M[j]) for i,j in combinations(range(k),2)] if k>1 else [0.]
    return dict(k=k, seed=run['seed'],
                best_val_loss=best_val_loss, n_epochs=n_epochs,
                n_active=n_active, active_frac=n_active/k,
                effective_k=effective_k, effective_k_frac=effective_k/k,
                diversity=float(np.mean(diffs)),
                assign_frac=float((W.max(axis=1)>=ASSIGN_THRESH).mean()),
                mean_entropy=entropy.mean())


def _cross_seed_stability(runs_k):
    if len(runs_k) < 2: return float('nan')
    sims = []
    for r1, r2 in combinations(runs_k, 2):
        M1 = r1['means'].values; M2 = r2['means'].values
        n1 = M1 / (np.linalg.norm(M1, axis=1, keepdims=True)+1e-9)
        n2 = M2 / (np.linalg.norm(M2, axis=1, keepdims=True)+1e-9)
        C  = n1 @ n2.T
        ri, ci = linear_sum_assignment(1-C)
        sims.append(float(C[ri,ci].mean()))
    return float(np.mean(sims))


all_runs, all_metrics = {}, []
for k in K_VALUES:
    dim_matches = [p for p in RUNS_DIR.iterdir()
                   if p.is_dir() and p.name.startswith('dim_')
                   and p.name.split('_')[-1].isdigit()
                   and int(p.name.split('_')[-1]) == k]
    if not dim_matches: print(f'  K={k}: not found, skipping'); continue
    dim_dir = dim_matches[0]
    runs_k = []
    for sd in sorted(p for p in dim_dir.iterdir() if p.is_dir() and p.name.startswith('seed_')):
        run = _load_run(k, int(sd.name.split('_')[1]))
        if run is None: print(f'  K={k} {sd.name}: incomplete'); continue
        runs_k.append(run); all_metrics.append(_compute_metrics(run))
    all_runs[k] = runs_k
    print(f'  K={k}: {len(runs_k)} seed(s) loaded')

metrics_df = pd.DataFrame(all_metrics)
print(f'\nTotal runs: {len(metrics_df)}')
display(metrics_df.round(4))

In [ ]:
stability    = {k: _cross_seed_stability(runs) for k, runs in all_runs.items()}
stability_df = pd.DataFrame([{'k':k,'cross_seed_stability':v} for k,v in stability.items()])

agg = metrics_df.groupby('k').agg(
    best_val_loss_mean=('best_val_loss','mean'), best_val_loss_std=('best_val_loss','std'),
    active_frac_mean=('active_frac','mean'), effective_k_mean=('effective_k','mean'),
    effective_k_frac_mean=('effective_k_frac','mean'), diversity_mean=('diversity','mean'),
    assign_frac_mean=('assign_frac','mean'), n_seeds=('seed','count'),
).reset_index().merge(stability_df, on='k', how='left')

print('Per-K summary:')
display(agg.round(4))

## Diagnostic Plots

| Panel | What to look for |
|---|---|
| Val loss | Elbow — marginal gain flattens |
| Active fraction | Should stay 1.0; drop = collapse |
| Effective K | Should track K; plateau = redundant archetypes |
| Diversity | Drop = new archetypes become near-duplicates |
| Cross-seed stability | High = robust; low = different random basins |
| Assignment concentration | Fraction of cells with a dominant archetype |

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ks  = agg['k'].values
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('K Selection Diagnostics — Breast CyTOF', fontsize=14, fontweight='bold')

def _plot(ax, col, col_std=None, color='steelblue', ylabel='', title=''):
    y = agg[col].values
    ax.plot(ks, y, 'o-', color=color, linewidth=2, markersize=6)
    if col_std and col_std in agg.columns:
        e = agg[col_std].fillna(0).values
        ax.fill_between(ks, y-e, y+e, color=color, alpha=0.15)
    ax.set_title(title); ax.set_xlabel('K'); ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

_plot(axes[0,0],'best_val_loss_mean','best_val_loss_std','steelblue','Val loss','Val Loss (best epoch)')
_plot(axes[0,1],'active_frac_mean',color='tomato',ylabel='Fraction active',title='Active Archetype Fraction')
axes[0,1].axhline(1.0,color='gray',linestyle='--',linewidth=1,alpha=0.6); axes[0,1].set_ylim(0,1.1)
axes[0,2].plot(ks,ks,'--',color='gray',linewidth=1,label='ideal')
_plot(axes[0,2],'effective_k_mean',color='mediumseagreen',ylabel='Effective K',title='Effective K vs K')
axes[0,2].legend(fontsize=8)
_plot(axes[1,0],'diversity_mean',color='darkorchid',ylabel='Mean pairwise L2',title='Archetype Diversity')
ax=axes[1,1]
if agg['cross_seed_stability'].notna().any():
    ax.plot(ks,agg['cross_seed_stability'].values,'o-',color='darkorange',linewidth=2,markersize=6)
    ax.set_ylim(0,1.05); ax.set_ylabel('Mean cosine similarity')
else:
    ax.text(0.5,0.5,'Only 1 seed per K\n(add more seeds for\nstability estimate)',
            ha='center',va='center',transform=ax.transAxes,fontsize=10,color='gray')
ax.set_title('Cross-Seed Stability'); ax.set_xlabel('K')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
_plot(axes[1,2],'assign_frac_mean',color='sienna',ylabel='Fraction assigned',
      title=f'Assignment Concentration (max_w >= {ASSIGN_THRESH})')
axes[1,2].set_ylim(0,1.1)
fig.tight_layout(); plt.show()

## Automated Recommendation

In [ ]:
import warnings
rec = agg.copy()
lr  = rec['best_val_loss_mean'].max() - rec['best_val_loss_mean'].min()
rec['score_loss']      = 1-(rec['best_val_loss_mean']-rec['best_val_loss_mean'].min())/lr if lr>0 else 1.0
rec['score_active']    = rec['active_frac_mean'].clip(0,1)
rec['score_eff_k']     = rec['effective_k_frac_mean'].clip(0,1)
rec['score_stability'] = rec['cross_seed_stability'].fillna(0.5).clip(0,1)
rec['composite_score'] = (0.35*rec['score_loss']+0.25*rec['score_active']+
                          0.25*rec['score_eff_k']+0.15*rec['score_stability'])
display(rec[['k','best_val_loss_mean','active_frac_mean','effective_k_frac_mean',
             'cross_seed_stability','composite_score']].sort_values('composite_score',ascending=False).round(4))
best_k = int(rec.loc[rec['composite_score'].idxmax(),'k'])
print(f'\n>>> Recommended K: {best_k} <<<')
br = rec[rec['k']==best_k].iloc[0]
if br['active_frac_mean']<1.0:
    warnings.warn(f'K={best_k} has ~{round((1-br["active_frac_mean"])*best_k)} dead archetype(s).')
if br['effective_k_frac_mean']<0.5:
    warnings.warn(f'K={best_k} effective utilisation low ({br["effective_k_frac_mean"]:.1%}).')

## Per-Archetype Inspection — Recommended K

For the breast dataset, archetypes reflect cell states (luminal/basal subtypes,
mesenchymal/EMT, immune, cycling) rather than named cell types.

In [ ]:
best_runs = all_runs.get(best_k, [])
if not best_runs:
    print(f'No runs for K={best_k}')
else:
    r = best_runs[0]
    means, mean_vars = r['means'], r['cvars'].mean(axis=1)
    print(f'K={best_k}, seed={r["seed"]} — archetype profiles (z-score means)\n')
    print(f'{"Arch":<6} {"MeanVar":>10}  {"Status":<8}  Top 4 (+)  |  Bottom 2 (-)')
    print('-'*100)
    for comp in means.index:
        idx  = int(comp.split('_')[1])
        row  = means.loc[comp]
        mvar = mean_vars[comp]
        top4 = row.nlargest(4); bot2 = row.nsmallest(2)
        print(f'A{idx:02d}    {mvar:10.2f}  {"ACTIVE" if mvar<VAR_THRESH else "DEAD":<8}  '
              f'{", ".join(f"{m}={v:+.2f}" for m,v in top4.items()):<55}  '
              f'{", ".join(f"{m}={v:+.2f}" for m,v in bot2.items())}')

## Archetype Mean Heatmap — Recommended K

In [ ]:
if best_runs:
    r = best_runs[0]
    means = r['means'].copy()
    means.index = [f'A{int(c.split("_")[1]):02d}' for c in means.index]
    vabs = max(abs(means.values.min()), abs(means.values.max()))
    fig, ax = plt.subplots(figsize=(max(10, len(means.columns)*0.45), max(4, best_k*0.5)))
    im = ax.imshow(means.values, aspect='auto', cmap='RdBu_r', vmin=-vabs, vmax=vabs)
    ax.set_xticks(range(len(means.columns)))
    ax.set_xticklabels(means.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(best_k)); ax.set_yticklabels(means.index, fontsize=9)
    ax.set_title(f'Archetype Means — K={best_k}  (z-score)', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.6, label='z-score')
    fig.tight_layout(); plt.show()